In [1]:
# 1. Eliminar rastro de instalaciones previas que causan conflictos
!apt-get remove -y google-chrome-stable chromium-browser chromium-chromedriver

# 2. Instalar Google Chrome Estable y dependencias de sistema
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get update
!apt-get install -y google-chrome-stable

# 3. Instalar Selenium
!pip install -U selenium webdriver-manager

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package google-chrome-stable
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 https://cli.github.com/packages stable InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,213 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-sec

In [11]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager

def descargar_pacs_forzado_extremo(url_objetivo):
    download_dir = os.path.join(os.getcwd(), "DESCARGAS_PACS_ALTA_RES")
    if not os.path.exists(download_dir): os.makedirs(download_dir)

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")

    # 🛡️ BYPASS DE SEGURIDAD TOTAL
    chrome_options.add_argument("--allow-running-insecure-content")
    chrome_options.add_argument("--ignore-certificate-errors")
    chrome_options.add_argument("--disable-web-security") # Desactiva la política de origen (CORS)
    chrome_options.add_argument(f"--unsafely-treat-insecure-origin-as-secure=http://186.4.147.51:18881")

    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "profile.default_content_setting_values.automatic_downloads": 1,
        "safebrowsing.enabled": False
    }
    chrome_options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 30)

    try:
        print("🚀 Iniciando Visor en Modo Forzado...")
        driver.get(url_objetivo)
        time.sleep(30) # Tiempo para que cargue Cornerstone

        # 1. Salto al Iframe
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        if iframes: driver.switch_to.frame(iframes[0])

        # 2. Localizar Series
        series = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.thumbnail, [data-cy*="thumbnail"]')))
        print(f"📦 Carpetas detectadas: {len(series)}")

        for s_idx in range(len(series)):
            series = driver.find_elements(By.CSS_SELECTOR, '.thumbnail, [data-cy*="thumbnail"]')
            print(f"\n📂 CARPETA {s_idx+1}")
            driver.execute_script("arguments[0].click();", series[s_idx])
            time.sleep(15)

            for img_idx in range(1, 301):
                print(f"   🖼️ Imagen {img_idx}:", end=" ")
                try:
                    # PASO 1: ACTIVAR VIEWPORT (Evita el error de 'Stacks')
                    driver.execute_script("""
                        var canvas = document.querySelector('canvas, .viewport-element');
                        if (canvas) {
                            var rect = canvas.getBoundingClientRect();
                            var x = rect.left + (rect.width/2);
                            var y = rect.top + (rect.height/2);
                            canvas.dispatchEvent(new MouseEvent('mousedown', {bubbles:true, clientX:x, clientY:y}));
                            canvas.dispatchEvent(new MouseEvent('mouseup', {bubbles:true, clientX:x, clientY:y}));
                        }
                    """)

                    # PASO 2: ABRIR MENÚ (Usando el selector por dibujo de flecha que detectamos)
                    menu = driver.execute_script("""
                        var targetD = "M286.935 69.377c-3.614-3.617-7.898-5.424-12.848-5.424";
                        var path = Array.from(document.querySelectorAll('path')).find(p => p.getAttribute('d').includes(targetD));
                        if (path) {
                            var btn = path.closest('.toolbar-button') || path.parentElement;
                            var r = btn.getBoundingClientRect();
                            btn.dispatchEvent(new MouseEvent('click', {view:window, bubbles:true, clientX:r.right-2, clientY:r.bottom-2}));
                            return true;
                        }
                        return false;
                    """)
                    if not menu: raise Exception("MenuNoHallado")
                    time.sleep(3)

                    # PASO 3: CLIC EN DESCARGAR
                    driver.execute_script("document.querySelector('[data-cy=\"download\"]').click();")
                    time.sleep(3)

                    # PASO 4: PONER 2000PX
                    driver.execute_script("""
                        var inputs = document.querySelectorAll('input[type="number"]');
                        inputs.forEach(i => {
                            i.value = '2000';
                            i.dispatchEvent(new Event('input', { bubbles: true }));
                            i.dispatchEvent(new Event('change', { bubbles: true }));
                        });
                    """)
                    time.sleep(2)

                    # PASO 5: CLIC FINAL AZUL
                    btn_confirmar = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Descargar')]")))
                    driver.execute_script("arguments[0].click();", btn_confirmar)

                    print("✅", end=" ")
                    time.sleep(15) # Tiempo para el BLOB

                    # Limpiar y siguiente
                    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()
                    time.sleep(2)
                    webdriver.ActionChains(driver).send_keys(Keys.ARROW_DOWN).perform()
                    time.sleep(2)

                except Exception as e:
                    print(f"❌ ({str(e)})")
                    driver.save_screenshot(f"error_s{s_idx+1}_i{img_idx}.png")
                    # Intentar rescatar la sesión cerrando modales
                    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()
                    break
    finally:
        print(f"\nFinalizado. Carpeta: {download_dir}")
        driver.quit()

descargar_pacs_forzado_extremo("http://186.4.147.51:18881/viewer?url=http://186.4.147.51:18882/api/values/1.2.840.113704.9.1000.16.0.20260317111135778")

🚀 Iniciando Visor en Modo Forzado...
📦 Carpetas detectadas: 38

📂 CARPETA 1
   🖼️ Imagen 1: ❌ (Message: 
Stacktrace:
#0 0x5997c515722a <unknown>
#1 0x5997c4b55ab9 <unknown>
#2 0x5997c4baa046 <unknown>
#3 0x5997c4baa281 <unknown>
#4 0x5997c4bf4f74 <unknown>
#5 0x5997c4bf2116 <unknown>
#6 0x5997c4b9d662 <unknown>
#7 0x5997c4b9e451 <unknown>
#8 0x5997c511ad8b <unknown>
#9 0x5997c511dc65 <unknown>
#10 0x5997c5107458 <unknown>
#11 0x5997c511e7f0 <unknown>
#12 0x5997c50ee1c0 <unknown>
#13 0x5997c5144168 <unknown>
#14 0x5997c5144305 <unknown>
#15 0x5997c5155c5e <unknown>
#16 0x7c05dc536ac3 <unknown>
)

📂 CARPETA 2
   🖼️ Imagen 1: ❌ (Message: 
Stacktrace:
#0 0x5997c515722a <unknown>
#1 0x5997c4b55ab9 <unknown>
#2 0x5997c4baa046 <unknown>
#3 0x5997c4baa281 <unknown>
#4 0x5997c4bf4f74 <unknown>
#5 0x5997c4bf2116 <unknown>
#6 0x5997c4b9d662 <unknown>
#7 0x5997c4b9e451 <unknown>
#8 0x5997c511ad8b <unknown>
#9 0x5997c511dc65 <unknown>
#10 0x5997c5107458 <unknown>
#11 0x5997c511e7f0 <unknown>
#12 0x


Finalizado. Carpeta: /content/DESCARGAS_PACS_ALTA_RES


KeyboardInterrupt: 